<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/rl-course-hf/blob/main/unit2/introduction_to_qlearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/fabiobento/rl-course-hf/blob/main/unit2/introduction_to_qlearning.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

Adaptado do [Hugging Face Deep Reinforcement Learning Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt)

# Introdução ao Q-Learning

## Introdução

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/thumbnail.jpg" alt="The RL process" width="100%">
<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Nesta unidade, **vamos nos aprofundar em um dos métodos de RL: _value-based methods_** e estudar nosso primeiro algoritmo de RL: **Q-Learning**.

Vamos também **implementar nosso primeiro agente de RL do zero**, um agente de Q-Learning, e treiná-lo em dois ambientes:

1. Frozen-Lake-v1 (versão _non-slippery_): onde nosso agente precisará **ir do estado inicial (S) ao estado final (G)** caminhando apenas sobre blocos congelados (F) e evitando buracos (H).
2. Um táxi autônomo: onde nosso agente precisará **aprender a navegar** por uma cidade para **transportar seus passageiros do ponto A ao ponto B**.



<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/envs.gif" alt="The RL process" width="100%">
<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Nessa unidade, vamos:

- Aprender sobre **value-based methods**.
- Aprender sobre as **diferenças entre Monte Carlo e Temporal Difference Learning**.
- Estudar e implementar **nosso primeiro algoritmo RL**: Q-Learning.

Esta unidade é **fundamental se você deseja trabalhar com Deep Q-Learning**: o primeiro algoritmo Deep RL que jogou jogos Atari e superou o nível humano em alguns deles (breakout, space invaders, etc).

Então, vamos começar! 🚀

## O que é RL? Uma breve revisão

Na RL, criamos um agente capaz de **tomar decisões inteligentes**.

Por exemplo, um agente que **aprende a jogar um videogame**.
Ou um agente da bolsa de valores que **aprende a maximizar seus lucros** decidindo **quais ações comprar e quando vendê-las**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit1/images/RL_process.jpg" alt="The RL process" width="100%">
<span style="font-size:80%">
O processo RL: um loop de state, action, reward e próximo state.

Fonte: <a href="http://incompleteideas.net/book/RLbook2020.pdf" target="_blank">Reinforcement Learning: An Introduction, Richard Sutton and Andrew G. Barto</a>
</span>

Para tomar decisões inteligentes, nosso agente aprenderá com o ambiente, **interagindo com ele por meio de tentativa e erro** e recebendo recompensas (positivas ou negativas) **como único feedback**.

Seu objetivo **é maximizar sua recompensa cumulativa esperada**(_expected cumulative reward_) devido à hipótese da recompensa.

**O processo de tomada de decisão do agente é chamado de _policy_ $\pi$**: dado um estado, uma _policy_ produzirá uma ação ou uma distribuição de probabilidade sobre as ações. Ou seja, dada uma observação do ambiente, uma _policy_ fornecerá uma ação (ou múltiplas probabilidades para cada ação) que o agente deve realizar.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/policy.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Nosso objetivo é encontrar uma _optimal policy_ $\pi$*, ou seja, uma _policy_ que leve à melhor recompensa cumulativa esperada.

E para encontrar essa _optimal policy_ (resolvendo assim o problema de RL), existem dois tipos principais de métodos de RL:

- _Policy-based methods_: **treinar a _policy_ diretamente** para aprender qual ação tomar em um determinado estado.
- _Value-based methods_: **treinar uma _value function_** para aprender **qual estado é mais valioso** e usar essa _value function_ **para executar a ação que leva a ele**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/two-approaches.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

E nesta unidade, **vamos nos aprofundar nos _value-based methods_**.

## Dois tipos de _value-based methods_

Nos _value-based methods_, **aprendemos uma _value function_** que **mapeia um estado para o valor esperado ao estar nesse estado**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/vbm-1.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

O valor de um estado é o **retorno descontado esperado** que o agente pode obter se **começar nesse estado e agir de acordo com nossa _policy_**.

>Mas o que significa agir de acordo com nossa _policy_? Afinal, não temos uma _policy_ em _value-based methods_, pois treinamos uma _value function_ e não uma _policy_.

Lembre-se de que o objetivo de um **agente RL é ter uma _optimal policy_ $\pi*$**.

Para encontrar a _optimal policy_, aprendemos sobre dois métodos diferentes:

- _Value-based methods_: **Indiretamente, treinando uma _value function_** que gera o valor de um estado ou de um par estado-ação. Com base nessa _value function_, nossa _policy_ tomará uma ação.

Como a _policy_ não é treinada/aprendida, **precisamos especificar seu comportamento**. Por exemplo, se quisermos uma _policy_ que, dada a _value function_, tome ações que sempre levem à maior recompensa, criaremos uma Política Gananciosa(_Greedy Policy_).

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/two-approaches-3.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Dado um estado, nossa action-value function (que treinamos) gera o valor de cada ação nesse estado. Então, nossa Greedy Policy pré-definida seleciona a ação que produzirá o maior valor, dado um estado ou um par de estado-ação.

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Consequentemente, qualquer que seja o método utilizado para resolver o problema, haverá uma _policy_.
No caso dos _value-based methods_, não se treina a _policy_: a _policy_ **é apenas uma função pré-definida simples** (por exemplo, a _Greedy policy_) que utiliza os valores fornecidos pela _value function_ para selecionar suas ações.

Portanto, a diferença é:
- No treinamento _policy-based_, **a _optimal policy_ (denotada por $\pi*$) é encontrada através do treinamento direto da _policy_**.
- No treinamento _value-based_, **encontrar uma _optimal value function_ (denotada por $Q*$ ou $V*$) leva a uma _optimal policy_**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/link-value-policy.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Na verdade, na maioria das vezes, em _value-based methods_, você usará uma **_Epsilon-Greedy policy_** que lida com o trade-off entre _exploration_/_expoitation_ ; falaremos sobre isso quando discutirmos o Q-Learning na segunda parte desta unidade.

Como mencionamos acima, temos dois tipos de _value-based functions_:
- _state-value function_
- _action-value function_

### _State-value function_

Escrevemos a _state-value function+ sob uma _policy_ $\pi$ desta forma:

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/state-value-function-1.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Para cada estado, a _state-value function_ gera o retorno esperado(_expected return_) se o agente **começar nesse estado** e seguir a _policy_ para sempre depois disso (para todos os futuros timesteps).

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/state-value-function-2.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Se considerarmos o estado com valor -7: é o retorno esperado a partir desse estado e tomando ações de acordo com nossa policy (greedy policy), então direita, direita, direita, para baixo, para baixo, direita, direita.


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

### _Action-value function_

Na _action-value function_, para cada par estado-ação, a _action-value function_ gera **o retorno esperado** se o agente começar nesse estado, realizar essa ação e seguir a _policy_ para sempre.

O valor de executar a ação $\alpha$ no estado $s$ sob a _policy_ $\pi$ é:

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/action-state-value-function-1.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/action-state-value-function-2.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Vemos que a diferença é:
- Para a _state-value function_, calcumamos **o valor do estado** $S_{t}$
- Para a _action-value function_ , calculamos **o valor do par _state-action_ $(S_{t},A_{t})$ e, portanto, o valor de executar a ação nesse estado**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/two-types.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Nota: Não preenchemos todos os pares state-action para o exemplo da action-value function.

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Em ambos os casos, independentemente da _function value_ que escolhermos (_state-value function_ ou _action-value function_), **o valor retornado é o retorno esperado (_expected return_)**.

No entanto, o problema é que, para calcular **CADA valor de um _state_ ou de um par _state-action_, precisamos somar todas as recompensas que um agente pode obter se começar nesse estado.**

Esse pode ser um processo computacionalmente caro, e **é aí que a equação de Bellman vem nos ajudar**.

## A Equação de Bellman: simplifique nossa estimativa de valor 

A equação de Bellman **simplifica o cálculo do _state value_ ou do _state-action value_**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/bellman.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Com o que aprendemos até agora, sabemos que se calcularmos $V(S_{t})$(o valor do estado), precisamos calcular o retorno esperado se começarmos naquele estado e seguirmos a _policy_ indefinidamente.**(A policy que definimos no exemplo a seguir é _Greedy Policy+; por simplicidade, então não teremos descontos nas recompensas).**

Então para calcular $V(S_{t})$, precisamos calcular a soma das as recompensas esperadas(_expected rewards_). Portanto:

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/bellman2.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Para calcular o valor do Estado 1: a soma das recompensas se o agente tivesse começado nesse estado e depois seguido a greedy policy (tomando ações que levam aos melhores valores de estado) para todos os passos temporais.

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Então, para calcular o $V(S_{t+1})$, precisamos calcular o retorno (_expected return_) se inicarmos a partir do estado $S_{t+1}$.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/bellman3.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Para calcular o valor do Estado 2: a soma das recompensas se o agente tivesse começado nesse estado e, em depois, seguido a policy durante todos os passos temporais.

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Como você deve ter notado, estamos repetindo o cálculo do valor de diferentes estados, o que pode ser tedioso se você precisar fazer isso para cada valor de estado ou valor de ação de estado.

Em vez de calcular o retorno esperado para cada estado ou cada par estado-ação, podemos usar a **equação de Bellman**. (dica: se você sabe o que é Programação Dinâmica, isso é muito semelhante! Se você não sabe o que é, não se preocupe!)

A equação de Bellman é uma equação recursiva que funciona assim: em vez de começar cada estado desde o início e calcular o retorno, podemos considerar o valor de qualquer estado $t$ como:

**A recompensa imediata $R_{t+1}$  + o valor descontado do estado que se segue $\gamma∗V(S_{t+1})$**

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/bellman4.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Se voltarmos ao nosso exemplo, podemos dizer que o valor do Estado 1 é igual ao retorno acumulado esperado(_expected cumulative return_) se começarmos nesse estado.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/bellman2.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Para calcular o valor do Estado 1: a soma das recompensas se **o agente começou nesse estado 1** e depois seguiu a **_policy_ durante todos os passos temporais**.

Isso é equivalente a $V(S_{t})$=recompensa imediata $R_{t+1}$ + o valor descontado do próximo estado $\gamma*V(S_{t+1})$

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/bellman6.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Para simplificar aqui não descontamos, então gamma=1

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Por uma questão de simplicidade, aqui não fazemos descontos, portanto $\gamma$ = 1. 

Mas você estudará um exemplo com gama = 0,99 na seção Q-Learning desta unidade.

- O valor de $V(S_{t+1})$= recompensa imediata $R_{t+2}$ + o valor descontado do próximo estado $(\gamma*V(S_{t+2}))$
- E assim por diante

Para recapitular, a ideia da equação de Bellman é que, em vez de calcular cada valor como a soma do retorno esperado, **o que é um processo demorado**, calculamos o valor como **a soma da recompensa imediata + o valor descontado do estado que se segue**.

Antes de passar para a próxima seção, pense sobre o papel do $\gamma$ na equação de Bellman.
- O que acontece se o valor do $\gamma$ for muito baixo (por exemplo, 0,1 ou mesmo 0)? O que acontece se o valor for 1? O que acontece se o valor for muito alto, como um milhão?

## Monte Carlo vs Temporal Difference Learning

A última coisa que precisamos discutir antes de mergulharmos no Q-Learning são as duas estratégias de aprendizagem.

Lembre-se de que um agente RL **aprende interagindo com seu ambiente**. A ideia é que, **dada a experiência e a recompensa recebida, o agente atualizará sua _value function_ ou _policy_**.

Monte Carlo e _Temporal Difference Learning_ são **duas estratégias diferentes sobre como treinar nossa _value function_ ou nossa _policy function_**. Ambas **usam a experiência para resolver o problema RL**.

Monte Carlo usa **um episódio inteiro de experiência antes de aprender**. A _Temporal Differences_, por outro lado, usa **apenas um passo($S_{t}$, $A_{t}$, $R_{t+1}$, $S_{t+1}$) para aprender**.

Vamos explicar ambos usando **usando apenas um exemplo com _value-based method_**.

### Monte Carlo: aprendizado ao final do episódio

Monte Carlo espera até o final do episódio, calcula $G_{t}$(o _return_) e usa-o como **um alvo para atualizar $V(S_{t})**$.

Por um lado, Monte Carlo usa um episódio inteiro de experiência antes de aprender. Por outro lado, a diferença temporal usa apenas um passo.

Portanto, é necessário **um episódio completo de interação antes de atualizar nossa função de valor**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/monte-carlo-approach.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Utilizaremos o exemplo da imagem abaixo:

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/MC-2.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

- Sempre começamos o episódio **no mesmo ponto de partida**.

- **O agente realiza ações usando a _policy_**. Por exemplo, usando uma estratégia _Epsilon Greedy_, uma _policy_ que alterna entre _exploration_ (ações aleatórias) e _exploitation_.

- Recebemos **a recompensa e o próximo estado**.

- Encerramos o episódio se o gato comer o rato ou se o rato se mover > 10 passos.

- No final do episódio, **temos uma lista de tuplas de Estados, Ações, Recompensas e Próximos Estados**(_**S**tate, **A**ctions, **R**ewards, Next **S**tates_). Por exemplo, [[Estado bloco 3 inferior, Ir para a esquerda, +1, Estado bloco 2 inferior], [Estado bloco 2 inferior, Ir para a esquerda, +0, Estado bloco 1 inferior]…].
- **O agente vai somar o total de recompensas $G_{t}$**(para verificar como se saiu).
- Ele então **atualiza $V(S_{t})$ baseado na seguinte fórmula**:

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/MC-3.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

- E **inicia um novo jogo com esse conhecimento**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/MC-3p.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Por exemplo, se treinarmos uma _state-value function_ usando Monte Carlo:

- Inicializamos nossa _value function_ **para que ela retorne o valor 0 para cada estado**

- Nossa taxa de aprendizado (_learning rate_-lr) é 0,1 e nossa taxa de desconto(_discount_) é 1 (ou seja, sem desconto)

- Nosso rato **explora o ambiente e realiza ações aleatórias**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/MC-4.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

- O rato deu mais de 10 passos, então o episódio termina.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/MC-4p.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

- Temos então uma lista de estado, ação, recompensa e próximo estado, **e precisamos calcular o retorno** com $G_{t}=R_{t+1}+R_{t+2}R_{t+3}\dots$.  Por simplicidade nao descontamos as recompensas nesse exemplo, então $G_{0}=R_{1}+R_{2}+R_{3}\dots=1+0+0+0+0+0+1+1+0+0$. Ou seja, $G_{0}=3$.

- Podemos calcular agora o novo $V(S_{0})$:

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/MC-5.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

$V(S_{0})=V(S_{0})+lr*[G_{0}-V(S_{0})]$  

$V(S_{0})=0+0,1*[3-0]$  

$V(S_{0})=0,3$


<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/MC-5p.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

### Temporal Difference Learning: aprendendo a cada passo

**_Temporal Difference_ espera por apenas uma iteração (um passo) $S_{t+1}$** para formar um alvo TD e atualiza $V(S_{t})$ usndo $R_{t+1}$ e $\gamma*V(S_{t+1})$

A idéia com **TD é atualizar o V(S_{t}) a cada passo**.

Mas, como não vivenciamos um episódio inteiro, não temos $G_{t}$ (retorno esperado). Em vez disso, **estimamos $G_{t}$​ somando $R_{t+1}$  e o valor descontado do próximo estado**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/TD-1.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Este método é chamado de TD(0) ou **_one-step_ TD (TD de um passo), pois atualiza a _value function_ após qualquer passo individual**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/TD-1p.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>